# Customer Lifetime Value (CLV) Prediction

## Hypothesis
- Customers who purchase more frequently, recently, and spend more in the past are likely to bring more revenue in the future.
- Demographic attributes like age, region, and gender influence purchasing behavior.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## Load Data

In [2]:
# Load dataset
df = pd.read_csv("synthetic_customer_transactions.csv")
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'])
df['Signup_Date'] = pd.to_datetime(df['Signup_Date'])

## Exploratory Data Analysis (EDA)

In [3]:
print("Date Range:", df['Transaction_Date'].min(), "to", df['Transaction_Date'].max())
print("Number of unique customers:", df['CustomerID'].nunique())

Date Range: 2021-01-01 00:00:00 to 2023-12-30 00:00:00
Number of unique customers: 10000


## Feature Engineering

In [7]:
# import pandas as pd

# Ensure date columns are in datetime format
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'])
df['Signup_Date'] = pd.to_datetime(df['Signup_Date'])

# Set new cutoff
cutoff_date = pd.Timestamp("2023-06-30")

# Split historical and prediction periods
df_pre = df[df['Transaction_Date'] <= cutoff_date]
df_post = df[(df['Transaction_Date'] > cutoff_date) & (df['Transaction_Date'] <= pd.Timestamp("2023-12-31"))]

# Aggregations
agg_funcs = {
    'Transaction_Date': [np.min, np.max, 'count'],
    'TotalPrice': np.sum,
    'Quantity': np.mean,
    'Discount_pct': np.mean,
    'Online_Spend': np.mean,
    'Offline_Spend': np.mean,
    'Returned': np.mean,
    'Channel': lambda x: x.mode()[0],
    'Payment_Method': lambda x: x.mode()[0]
}

customer_features = df_pre.groupby('CustomerID').agg(agg_funcs)
customer_features.columns = [
    'first_purchase', 'last_purchase', 'frequency', 'past_total_revenue',
    'avg_quantity', 'avg_discount', 'avg_online_spend', 'avg_offline_spend',
    'return_rate', 'most_used_channel', 'most_used_payment'
]

customer_features['recency'] = (cutoff_date - customer_features['last_purchase']).dt.days
customer_features['tenure'] = (customer_features['last_purchase'] - customer_features['first_purchase']).dt.days

# Demographics
demo_cols = ['CustomerID', 'Gender', 'Age', 'Region', 'Signup_Date', 'Customer_Segment']
demo_df = df[demo_cols].drop_duplicates(subset='CustomerID').set_index('CustomerID')
customer_features = customer_features.join(demo_df)

# Target CLV
df_post_clv = df_post.groupby('CustomerID')['TotalPrice'].sum().rename('base_clv')

# Final dataset
final_df = customer_features.join(df_post_clv)
final_df = final_df.dropna(subset=['base_clv'])

# Check shape
final_df.shape


C:\Users\aufaa\AppData\Local\Temp\ipykernel_17496\4068357998.py:27: FutureWarning: The provided callable <function min at 0x0000025FC80845E0> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  customer_features = df_pre.groupby('CustomerID').agg(agg_funcs)
C:\Users\aufaa\AppData\Local\Temp\ipykernel_17496\4068357998.py:27: FutureWarning: The provided callable <function max at 0x0000025FC80844A0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  customer_features = df_pre.groupby('CustomerID').agg(agg_funcs)
C:\Users\aufaa\AppData\Local\Temp\ipykernel_17496\4068357998.py:27: FutureWarning: The provided callable <function sum at 0x0000025FC805BA60> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be use

(9321, 19)

In [12]:
final_df.columns, final_df.head()

(Index(['first_purchase', 'last_purchase', 'frequency', 'past_total_revenue',
        'avg_quantity', 'avg_discount', 'avg_online_spend', 'avg_offline_spend',
        'return_rate', 'most_used_channel', 'most_used_payment', 'recency',
        'tenure', 'Gender', 'Age', 'Region', 'Signup_Date', 'Customer_Segment',
        'base_clv'],
       dtype='object'),
            first_purchase last_purchase  frequency  past_total_revenue  \
 CustomerID                                                               
 CUST_00000     2021-04-27    2023-05-21          8             3387.06   
 CUST_00001     2021-03-29    2023-04-21          7             4361.97   
 CUST_00002     2021-01-25    2023-06-30         15             8053.26   
 CUST_00003     2021-03-13    2023-06-23         24            15551.56   
 CUST_00004     2021-01-13    2022-10-24         13             7870.65   
 
             avg_quantity  avg_discount  avg_online_spend  avg_offline_spend  \
 CustomerID                      

## Model Training

In [11]:
# Drop datetime and target columns
X = final_df.drop(columns=['first_purchase', 'last_purchase', 'Signup_Date', 'base_clv'])

# One-hot encoding for categorical features
X = pd.get_dummies(X, drop_first=True)

# Target variable
y = final_df['base_clv']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define models
models = {
    'LinearRegression': LinearRegression(),
    'DecisionTree': DecisionTreeRegressor(random_state=42),
    'RandomForest': RandomForestRegressor(random_state=42),
    'XGBoost': XGBRegressor(random_state=42, objective='reg:squarederror')
}

# Train and evaluate models
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results[name] = {
        'MAE': mean_absolute_error(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'R2': r2_score(y_test, y_pred)
    }

results_df = pd.DataFrame(results).T
results_df


,MAE,RMSE,R2
LinearRegression,767.073118,971.328957,-0.003738
DecisionTree,1136.601823,1459.845040,-1.267259
RandomForest,788.699858,992.973928,-0.048971
XGBoost,841.038907,1070.506908,-0.219177


In [ ]:
final_df.shape

(9321, 19)

## Evaluation

In [ ]:
results_df = pd.DataFrame(results).T
print(results_df)

## Iteration
- Try hyperparameter tuning
- Add more time-based or behavioral features
- Consider segmenting customers before modeling